# 17 - Collect a CartPole Dataset

This notebook collects CartPole the same way `01_collect_dataset.ipynb` collects FrozenLake:

1. Build environments with `mouse_gym.EnvConfig` and `make_group_env`.
2. Create one `Datastore` for each environment stream.
3. Step the grouped environment and append each flat row.
4. Publish the stores with `push_stores_to_hub`.

CartPole's observation is a length-4 float vector, not a discrete id. The row stores that vector as `observation`. `18_train_offline_cartpole_dqn.ipynb` reads each element with a text field `input_index`.

The four elements, in Gymnasium order, are cart position, cart velocity, pole angle (radians), and pole angular velocity. Actions are discrete: `0` or `1`. There is no optimal-Q field, so this collector takes a random action every step.

`MAX_EPISODE_STEPS` is `50` and `MAX_TASK_EPISODES` is `4`, shorter than CartPole-v1's usual 500-step episode, so one task stays inside the training notebook's token budget. A step that keeps the pole up scores `1`. The episode ends when the pole falls, the cart leaves the track, or the step cap is hit.

This is a short usage example, not a full experiment. Scale collection up in [mouse-experiment](https://github.com/micahr234/mouse-experiment).


In [ ]:
import gymnasium as gym
import numpy as np

from mouse_gym import EnvConfig, make_group_env
from repeat_task_gym import RepeatTaskEnv
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID = "mouse-example-cartpole-dataset"  # Hugging Face dataset repo for push_stores_to_hub
NUM_ENVS = 4                                    # number of environment streams in the GroupEnv
STEPS_PER_ENV = 256                             # rows collected for each datastore
MAX_EPISODE_STEPS = 50                          # max steps per episode (shorter than CartPole-v1's 500)
MAX_TASK_EPISODES = 4                           # max episodes per task


## Build Environment

CartPole is a Gymnasium env. `RepeatTaskEnv` groups `MAX_TASK_EPISODES` episodes into one task, and `EnvConfig.env_fn` turns that into a mouse-gym stream. `seed_mode="stream"` gives each task a new seed. `episode_seed_mode="per_episode"` reseeds every inner episode.

`step_row` stores the length-4 observation vector, the mouse-gym task flags `terminated` and `truncated`, and the inner episode flags from the observation. `task_index` advances when the task ends.


In [ ]:
def make_cartpole(*, stream: int):
    def env_fn():
        pole = gym.make("CartPole-v1", max_episode_steps=MAX_EPISODE_STEPS)
        return RepeatTaskEnv(
            pole,
            max_task_episodes=MAX_TASK_EPISODES,
            episode_seed_mode="per_episode",
        )
    return env_fn


def new_cursor(*, n: int) -> dict:
    return {
        "task_index": [0] * n,
        "episode_index": [0] * n,
        "episode_start": [True] * n,
    }


def step_row(*, env, index: int, action, output, cursor: dict) -> dict:
    observation, reward, terminated, truncated, info = output
    episode_terminated = bool(observation["episode_terminated"])
    episode_truncated = bool(observation["episode_truncated"])
    row = {
        "action": int(action),
        "observation": observation["observation"],
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "episode_terminated": episode_terminated,
        "episode_truncated": episode_truncated,
        "episode_ended": episode_terminated or episode_truncated,
        "episode_index": cursor["episode_index"][index],
        "episode_start": cursor["episode_start"][index],
        "task_index": cursor["task_index"][index],
        "step_index": env.envs[index].step_index,
    }
    if terminated or truncated:
        cursor["task_index"][index] += 1
        cursor["episode_index"][index] = 0
        cursor["episode_start"][index] = True
    elif episode_terminated or episode_truncated:
        cursor["episode_index"][index] += 1
        cursor["episode_start"][index] = True
    else:
        cursor["episode_start"][index] = False
    return row


configs = [
    EnvConfig(
        name=f"cartpole_{i}",
        seed=i,
        seed_mode="stream",
        env_fn=make_cartpole(stream=i),
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)
cursor = new_cursor(n=len(env.names))


## Create Datastores

One `Datastore` per environment stream. Each store keeps that stream's rows in order. `DataLoader` mixes streams later.


In [ ]:
stores = [Datastore(name=name) for name in env.names]  # one ordered store per env stream


## Collect Transitions

Each step merges the action you sent with the environment output. `observation` stays a length-4 vector. CartPole's `info` is empty, so it is dropped and the row stays flat.

The first `env.step` also resets each stream. That row is stored, so every stream starts at its first observed state. Reward on that reset frame is `0`. Later steps score `1` while the pole is up.


In [ ]:
def run_rollout(*, env, stores: list, steps_per_env: int) -> None:
    """Step environments at random and append flat rows into per-stream datastores."""
    env.metrics.clear()
    for step in range(steps_per_env):
        actions = env.action_space.sample()
        outputs = env.step(actions)
        for i, output in enumerate(outputs):
            row = step_row(
                env=env, index=i, action=actions[i], output=output, cursor=cursor,
            )
            stores[i].append(data=row)
        if (step + 1) % 64 == 0 or (step + 1) == steps_per_env:
            task_scores = [
                r for env_tasks in env.metrics.episode_cum_rewards for r in env_tasks
            ]
            mean_task_score = (
                sum(task_scores) / len(task_scores) if task_scores else float("nan")
            )
            print(
                f"step {step + 1}/{steps_per_env} | "
                f"{len(task_scores)} tasks | mean task return {mean_task_score:.1f}"
            )
            env.metrics.clear()

run_rollout(env=env, stores=stores, steps_per_env=STEPS_PER_ENV)
env.close()


## Push to the Hub

`push_stores_to_hub` uploads every datastore to one Hugging Face dataset repository. Each store is one config. `18_train_offline_cartpole_dqn.ipynb` reloads them with `load_stores_from_hub`.


In [ ]:
url = push_stores_to_hub(stores=stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")
